In [1]:
# -------------- ETAPA 1: CARGA DOS DADOS DE SAÚDE (RESPIRATÓRIAS) ---------------------
import pandas as pd
import numpy as np

# Pasta onde salvei os CSVs do DATASUS
pasta = r'C:\Users\fabio\TCC\13_internacao_resp'
arquivo_antigo = f'{pasta}\\Intern_resp_1995_2007.csv'
arquivo_novo = f'{pasta}\\Intern_resp_2008_2025.csv'

print("--- Carregando arquivos de internação ---")

# 1. Lendo a parte antiga (95 a 2007)
# Usei latin1 porque o governo sempre manda com esse encoding por causa dos acentos
df_part1 = pd.read_csv(arquivo_antigo, sep=';', encoding='latin1')
print(f"Parte 1 carregada: {df_part1.shape}")

# 2. Lendo a parte nova (2008 a 2025)
df_part2 = pd.read_csv(arquivo_novo, sep=';', encoding='latin1')
print(f"Parte 2 carregada: {df_part2.shape}")

# 3. Conferindo as primeiras linhas pra ver se o separador ';' funcionou
print("\nExemplo rápido (Parte 1):")
print(df_part1.head(2))

print("\nExemplo rápido (Parte 2):")
print(df_part2.head(2))

--- Carregando arquivos de internação ---
Parte 1 carregada: (853, 11)
Parte 2 carregada: (853, 19)

Exemplo rápido (Parte 1):
                    Município 1998  1999  2000  2001  2002  2003  2004  2005  \
0  310010 ABADIA DOS DOURADOS  105    69    56    80    96    81    63    57   
1               310020 ABAETE  368   321   400   362   423   391   383   350   

   2006  2007  
0    54    70  
1   406   353  

Exemplo rápido (Parte 2):
                    Município 2008 2009 2010 2011 2012 2013 2014 2015 2016  \
0  310010 ABADIA DOS DOURADOS   33   34   29   17   31   53   37   51   26   
1               310020 ABAETE  375  285  306  369  372  360  313  313  343   

  2017 2018 2019 2020 2021 2022 2023 2024 2025  
0   29   22    -    -    -    -    -    -   30  
1  225  214  267  169  145  292  342  385  310  


In [2]:
# -------------- ETAPA 2: LIMPANDO E EMPILHANDO AS DATAS ---------------------

# Criei essa lista pra rodar a mesma limpeza nos dois arquivos de uma vez
dfs_para_limpar = [df_part1, df_part2]
dfs_prontos = []

print("--- Iniciando o tratamento das tabelas ---")

for df in dfs_para_limpar:
    # 1. Pegando só o código IBGE que vem antes do nome da cidade (ex: 310400)
    df['id_municipio_6'] = df['Município'].astype(str).str.split(' ').str[0]
    
    # 2. Tirando a coluna de nome original e a coluna de 'Total' pra não sujar o melt
    cols_drop = ['Município']
    if 'Total' in df.columns:
        cols_drop.append('Total')
    
    df_limpo = df.drop(columns=cols_drop, errors='ignore')
    
    # 3. Transformando as colunas de Ano em linhas (formato Long)
    # id_ano vai guardar o ano e bruto_internacoes_resp guarda o valor
    df_long = df_limpo.melt(
        id_vars=['id_municipio_6'],
        var_name='id_ano',
        value_name='bruto_internacoes_resp'
    )
    
    dfs_prontos.append(df_long)

# 4. Juntando a parte antiga (95-07) com a nova (08-25) em um tabelão só
df_final = pd.concat(dfs_prontos, ignore_index=True)

print(f"Base unificada com {len(df_final)} linhas.")
print(df_final.head())

--- Iniciando o tratamento das tabelas ---
Base unificada com 23884 linhas.
  id_municipio_6 id_ano bruto_internacoes_resp
0         310010   1998                    105
1         310020   1998                    368
2         310030   1998                    225
3         310040   1998                     39
4         310050   1998                    140


In [3]:
# -------------- ETAPA 3: TRATAMENTO FINAL E EXPORTAÇÃO (SAÚDE RESPIRATÓRIA) ---------------------
print("--- Finalizando a base de internações ---")

# 1. Tirando os hifens (-) do DATASUS e transformando em número real
# errors='coerce' é o que faz a mágica de virar NaN onde tem lixo
df_final['bruto_internacoes_resp'] = pd.to_numeric(df_final['bruto_internacoes_resp'], errors='coerce')

# 2. Onde não teve internação (NaN), o valor real é 0
df_final['bruto_internacoes_resp'] = df_final['bruto_internacoes_resp'].fillna(0)

# 3. Limpeza final no ID e no Ano pra não ter espaço sobrando
df_final['id_municipio_6'] = df_final['id_municipio_6'].astype(str).str.strip()
df_final['id_ano'] = df_final['id_ano'].astype(str).str.strip()

# 4. Filtrar a linha "Total" que o DATASUS sempre joga no fim do arquivo
# (Se o ID não tiver 6 dígitos, é lixo ou totalizador)
df_final = df_final[df_final['id_municipio_6'].str.len() == 6]

# 5. Ordenar por cidade e ano pra deixar o painel organizado (essencial pro Stata/R)
df_final.sort_values(by=['id_municipio_6', 'id_ano'], inplace=True)

# 6. Salvando na pasta de prontos
caminho_salvar = r'C:\Users\fabio\TCC\VARIAVEIS\13_intern_resp_FINAL.csv'
df_final.to_csv(caminho_salvar, index=False)

print(f"Sucesso! Arquivo 13 salvo em: {caminho_salvar}")

# Conferência rápida de Araxá pra ver se os anos estão batendo (95 até 2024/25)
print("\n--- Check Araxá (310400) ---")
print(df_final[df_final['id_municipio_6'] == '310400'].tail(10))

--- Finalizando a base de internações ---
Sucesso! Arquivo 13 salvo em: C:\Users\fabio\TCC\VARIAVEIS\13_intern_resp_FINAL.csv

--- Check Araxá (310400) ---
      id_municipio_6 id_ano  bruto_internacoes_resp
15397         310400   2016                   326.0
16250         310400   2017                   393.0
17103         310400   2018                   473.0
17956         310400   2019                   468.0
18809         310400   2020                   323.0
19662         310400   2021                   357.0
20515         310400   2022                   546.0
21368         310400   2023                   637.0
22221         310400   2024                   797.0
23074         310400   2025                   769.0
